In [ ]:
pip install openai

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="여기에 Api 키를 입력"
)

response = client.responses.create(
    model="gpt-5-mini",
    input="블라인드채용 주의사항 5가지 간략한 문장으로 짧게"
)

print(response.output_text)

1. 학력·출신·사진 등 개인정보는 가리세요.
2. 평가 기준을 사전에 명확히 정하세요.
3. 면접 질문으로 편견을 유발하지 마세요.
4. 심사위원에게 편향 교육을 실시하세요.
5. 개인정보보호와 관련 법규를 준수하세요.


In [ ]:
import openpyxl

file_path = r"여기에 작업경로를 설정"

wb = openpyxl.load_workbook(file_path)

print(wb.sheetnames)

ws = wb["자기소개서"]

for row in ws.iter_rows(min_row=1, max_row=1, values_only=True):
    print(row)

['자기소개서', '직무내용', '평가요소']
('면접번호', '이름', '모집분야', '전형단계', '질문', '답변')


In [ ]:
# -*- coding: utf-8 -*-

import json
import os
import time

import openpyxl
from openai import OpenAI

MODEL = "gpt-5-mini"

# -------------------------------------------------
# GPT 연결
# -------------------------------------------------

# 방법 1) 환경변수 OPENAI_API_KEY 를 사용 (권장)

# 방법 2) 코드에 직접 키를 넣고 싶다면 아래 줄의 주석을 풀고 키를 입력하세요.
API_KEY = "여기에 api 키를 입력"

if not API_KEY:
    raise RuntimeError(
        "OPENAI_API_KEY 가 설정되어 있지 않습니다.\n"
        "1) 환경변수 OPENAI_API_KEY를 설정한 뒤 '새' 터미널/커널에서 다시 실행하거나,\n"
        "2) 코드 상단의 API_KEY = os.getenv(...) 줄을 API_KEY = \"sk-...\" 형태로 직접 입력하세요."
    )

client = OpenAI(api_key=API_KEY)

# -------------------------------------------------
# 데이터 읽기
# -------------------------------------------------

def load_data(input_path):

    wb = openpyxl.load_workbook(
        input_path,
        data_only=True
    )

    ws_intro = wb["자기소개서"]
    ws_job = wb["평가요소"]

    candidates = {}

    for row in ws_intro.iter_rows(
            min_row=2,
            values_only=True):

        if row[0] is None:
            continue

        exam_no = row[0]
        name = row[1]
        field = row[2]
        status = row[3]
        question = row[4]
        answer = row[5]

        if exam_no not in candidates:

            candidates[exam_no] = {
                "면접번호": exam_no,
                "이름": name,
                "모집분야": field,
                "전형단계": status,
                "qa": []
            }

        candidates[exam_no]["qa"].append(
            {
                "question": question,
                "answer": answer
            }
        )

    job_info = {}

    for row in ws_job.iter_rows(
            min_row=2,
            values_only=True):

        if row[0] is None:
            continue

        field = row[0]
        unit_type = row[1]
        content = row[2]

        job_info.setdefault(field, {})
        job_info[field].setdefault(unit_type, [])

        if content:
            job_info[field][unit_type].append(
                str(content)
            )

    return candidates, job_info


# -------------------------------------------------
# 평가요소 읽기
# -------------------------------------------------

def load_evaluation_criteria(input_path):
    """'평가요소' 시트를 읽어 모집분야별 평가요소 정의를 반환.

    시트가 없거나 형식이 다르면 빈 dict를 반환하여
    전체 스크립트가 오류 없이 동작하도록 한다.

    기대하는 시트 구조 (헤더 포함):
        모집분야 | 평가요소 | 내용
    예)
        도핑검사법운용 | 책임감     | ...설명...
        도핑검사법운용 | 전문지식   | ...설명...
        도핑검사법운용 | 품행/인성  | ...설명...
        도핑검사법운용 | 발전가능성 | ...설명...
    """
    wb = openpyxl.load_workbook(
        input_path,
        data_only=True
    )

    if "평가요소" not in wb.sheetnames:
        return {}

    ws = wb["평가요소"]

    eval_info = {}

    for row in ws.iter_rows(
            min_row=2,
            values_only=True):

        if row[0] is None:
            continue

        field = row[0]
        factor = row[1] if len(row) > 1 else None
        content = row[2] if len(row) > 2 else None

        eval_info.setdefault(field, {})
        eval_info[field].setdefault(factor, [])

        if content:
            eval_info[field][factor].append(
                str(content)
            )

    return eval_info


# -------------------------------------------------
# 직무내용 / 평가요소 텍스트 생성
# -------------------------------------------------

def build_job_description(field, job_info):

    if field not in job_info:
        return ""

    lines = []

    for unit_type, contents in job_info[field].items():

        lines.append(f"[{unit_type}]")

        for c in contents:
            lines.append(f"- {c}")

    return "\n".join(lines)


def build_evaluation_description(field, eval_info):

    if field not in eval_info:
        return ""

    lines = []

    for factor, contents in eval_info[field].items():

        lines.append(f"[{factor}]")

        for c in contents:
            lines.append(f"- {c}")

    return "\n".join(lines)


# -------------------------------------------------
# GPT 호출
# -------------------------------------------------

def generate_questions(candidate, job_description, eval_description):

    intro_text = ""

    for idx, qa in enumerate(candidate["qa"], start=1):

        intro_text += f"""

[문항 {idx}]
질문:
{qa["question"]}

답변:
{qa["answer"]}
"""

    system_prompt = """
당신은 공공기관 블라인드 채용전문 면접관이다.
반드시 JSON만 출력한다.

형식:

{
 "main_questions":[
   {
     "question":"",
     "source_quote":"",
     "reason":"",
     "sub_questions":[
       "",
       "",
       ""
     ]
   }
 ]
}

규칙:
- 주질문은 5개 생성하며, 각 주질문의 평가목적은 아래와 같이 고정한다.
  - 주질문 1: 정신자세를 파악할 수 있는 질문
  - 주질문 2: 전문지식을 파악할 수 있는 질문
  - 주질문 3: 발표력을 파악할 수 있는 질문
  - 주질문 4: 품행을 파악할 수 있는 질문
  - 주질문 5: 발전가능성을 파악할 수 있는 질문
- 각 평가요소의 질문예시는 사용자 메시지의 "평가요소" 항목을 참조, 해당 평가요소에 대한 개별 맞춤화 질문을 구성.
- 각 주질문당 보조질문은 3개 생성한다. 보조질문은 해당 주질문(평가요소)을
  더 깊이 검증할 수 있는 꼬리질문(구체적 사례, 행동, 결과, 등)으로 구성.
- 경험기반면접(BEI) 스타일로 만들며, 질문의 내용은 현장에서 운용하기 쉽게 간결하게 구성. 
- 블라인드 채용 위배요소가 될 수 있는 구절이나 질문은 제한
- "reason"에는 해당 질문이 어떤 평가요소(책임감/전문지식/품행 및 인성역량/발전가능성)를
  어떻게 검증하려는 것인지 명시.
- "source_quote"는 해당 주질문을 만든 근거가 된 자기소개서 답변 속 문장/구절을
  "원문 그대로" 가능한 짧게(1~2문장 이내) 인용한다.
- 직무내용에서 도출된 질문이라 자기소개서에 직접적인 근거 문장이 없는 경우에는
  source_quote를 빈 문자열("")로 둔다.
- source_quote는 자기소개서 답변 본문에 실제로 존재하는 표현을 그대로 사용해야 한다
  (요약하거나 새로 만들어내지 말 것).
"""

    user_prompt = f"""
직무내용:

{job_description}

평가요소:

{eval_description}

자기소개서:

{intro_text}
"""

    response = client.responses.create(
        model=MODEL,
        input=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    return response.output_text


# -------------------------------------------------
# 결과 저장
# -------------------------------------------------

def save_result(output_path, results):

    wb = openpyxl.Workbook()

    ws = wb.active

    ws.title = "면접질문"

    headers = [
        "면접번호",
        "이름",
        "모집분야",
        "주질문",
        "근거구절",
        "질문이유",
        "보조질문"
    ]

    ws.append(headers)

    for candidate, result in results:

        try:

            data = json.loads(result)

            for q in data["main_questions"]:

                sub_text = "\n".join(
                    q["sub_questions"]
                )

                ws.append([
                    candidate["면접번호"],
                    candidate["이름"],
                    candidate["모집분야"],
                    q["question"],
                    q.get("source_quote", ""),
                    q["reason"],
                    sub_text
                ])

        except Exception as e:

            print(
                candidate["이름"],
                "파싱 실패:",
                e
            )

    wb.save(output_path)


# -------------------------------------------------
# 메인
# -------------------------------------------------

def main():

    input_file = r"자기소개서 원본파일을 입력"
    output_file = r"면접질문 생성결과 경로를 입력할 것"

    candidates, job_info = load_data(
        input_file
    )

    eval_info = load_evaluation_criteria(
        input_file
    )

    results = []

    total = len(candidates)

    for idx, candidate in enumerate(
            candidates.values(),
            start=1):

        print(
            f"[{idx}/{total}] "
            f"{candidate['이름']} 처리중"
        )

        job_desc = build_job_description(
            candidate["모집분야"],
            job_info
        )

        eval_desc = build_evaluation_description(
            candidate["모집분야"],
            eval_info
        )

        try:

            result = generate_questions(
                candidate,
                job_desc,
                eval_desc
            )

            results.append(
                (
                    candidate,
                    result
                )
            )

            print("완료")

        except Exception as e:

            print("실패:", e)

        time.sleep(1)

    save_result(
        output_file,
        results
    )

    print("모든 작업 완료")


if __name__ == "__main__":
    main()

[1/48] BLIND 처리중
완료
[2/48] BLIND 처리중
완료
[3/48] BLIND 처리중
완료
[4/48] BLIND 처리중
완료
[5/48] BLIND 처리중
완료
[6/48] BLIND 처리중
완료
[7/48] BLIND 처리중
완료
[8/48] BLIND 처리중
완료
[9/48] BLIND 처리중
완료
[10/48] BLIND 처리중
완료
[11/48] BLIND 처리중
완료
[12/48] BLIND 처리중
완료
[13/48] BLIND 처리중
완료
[14/48] BLIND 처리중
완료
[15/48] BLIND 처리중
완료
[16/48] BLIND 처리중
완료
[17/48] BLIND 처리중
완료
[18/48] BLIND 처리중
완료
[19/48] BLIND 처리중
완료
[20/48] BLIND 처리중
완료
[21/48] BLIND 처리중
완료
[22/48] BLIND 처리중
완료
[23/48] BLIND 처리중
완료
[24/48] BLIND 처리중
완료
[25/48] BLIND 처리중
완료
[26/48] BLIND 처리중
완료
[27/48] BLIND 처리중
완료
[28/48] BLIND 처리중
완료
[29/48] BLIND 처리중
완료
[30/48] BLIND 처리중
완료
[31/48] BLIND 처리중
완료
[32/48] BLIND 처리중
완료
[33/48] BLIND 처리중
완료
[34/48] BLIND 처리중
완료
[35/48] BLIND 처리중
완료
[36/48] BLIND 처리중
완료
[37/48] BLIND 처리중
완료
[38/48] BLIND 처리중
완료
[39/48] BLIND 처리중
완료
[40/48] BLIND 처리중
완료
[41/48] BLIND 처리중
완료
[42/48] BLIND 처리중
완료
[43/48] BLIND 처리중
완료
[44/48] BLIND 처리중
완료
[45/48] BLIND 처리중
완료
[46/48] BLIND 처리중
완료
[47/48] BLIND 처리중
완료
[48/48] BLIND 처리중
완료
모